# Visualize Model Predictions

This notebook loads a trained model and visualizes its predictions on a new sample.

In [ ]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
from plot_utilities import *
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
import time
from IPython.display import display
import plotly.express as px

In [ ]:
def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / id
    with open(model_path, 'r') as f:
        config = json.load(f)
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    return model
    

In [ ]:
# choose data
model_type = 'regular'


# ---------------------------------------------------------------------------
sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
#from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
from utilities_era5 import ERA5_Graph_Generator as Data_Generator



# Paths
trained_models_path = project_root / 'ERA5' / 'trained_models' / model_type
training_logs_path = trained_models_path / 'logs'
data_path = project_root.parent / 'ERA5' / 'data'  # Adjust if needed

print('Available models:\n')
files = sorted(os.listdir(trained_models_path), reverse=True)

for i, f in enumerate(files):
    if (Path(f).suffix == ".pth"):
        print(f'[{i}] {f}')


## 2. Load Model and Configuration

In [ ]:
# Set model ID
model_id = 'regular_papermodel2_2026-05-04_t12-01-43'
# model_id = files[19]#'reg_medium_woPos_deep6-regular-2026-04-13_t18-58-47'



# ---------------------------------------------------------------------------
if model_id.endswith('.pth'):
    model_id = model_id[:-4]

# Load full configuration (includes model, dataset, and training configs)
config_path = f'{training_logs_path}/configs/{model_id}.json'
with open(config_path, 'r') as f:
    config = json.load(f)

# Load loss records
loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')

print('Configuration:')
for key, value in config.items():
    print(f'  {key}: {value}')

print(f'\Training completed {len(loss_df)} epochs')
print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}')
print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')

In [ ]:
1e-04

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(loss_df['epoch'], loss_df['loss'], label='Training Loss', marker='o')
ax.plot(loss_df['epoch'], loss_df['val_loss'], label='Validation Loss', marker='s')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title(f'Training History - {model_id}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [ ]:
model = KernelNN(
    width=config['width'],
    ker_width=config['ker_width'],
    depth=config['depth'],
    ker_in=config['edge_attr'],
    in_width=config['node_features'],
    out_width=config['out_width']
).to(device)

# Load trained weights
model_path = f'{trained_models_path}/{model_id}.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print(f'Model loaded: {model_id}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Create New Sample

Use the same functionality as dataset creation to generate a single sample.

In [ ]:
date = "2019-01-20"

#bbox = lon_min, lat_min, lon_max, lat_max
bbox = -12, 30, 33, 75
#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')

N=10000
print('generate data')
graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
sample = graph.to(device)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})

In [ ]:
print('predict results')
output = model(sample).detach().cpu().numpy()

---
Plot

In [ ]:
locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})

res_model = pd.DataFrame(
    {
        "x1": output[:, 0],
        "x2": output[:, 1],
        "x3": output[:, 2],
    }
)
res_model = pd.concat([locations, res_model], axis=1)

figsize = (5, 5)
fig = rgb(res_model, figsize=figsize)
#fig = rgb_scale(res_model, percentile=1, figsize=figsize)

In [ ]:
fig_3d = px.scatter_3d(
        res_model,
        x="x1",
        y="x2",
        z="x3",
        color="x3",  # optional coloring
        title=f"3D Scatter: {model_id}"
    )
fig_3d.update_traces(marker=dict(size=1)) 
fig_3d.show()

In [ ]:

# example input (replace with your array)
X = sample.x.detach().cpu().numpy() # shape: (samples, features)

# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})


# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations, res_pca], axis=1)

fig = rgb(res_pca, figsize=figsize)

In [ ]:
fig_3d = px.scatter_3d(
        res_pca,
        x="x1",
        y="x2",
        z="x3",
        color="x3",  # optional coloring
        title=f"3D Scatter: PCA"
    )
fig_3d.update_traces(marker=dict(size=1)) 
fig_3d.show()

## Compare models

In [ ]:
from IPython.display import display
import time

In [ ]:
for i, f in enumerate(files):
    if Path(f).suffix == ".pth":
        print(f'[{i}] {f}')

In [ ]:
for i in [1,3,5,7,9,11,13,15]:
    model_id = files[i]#'reg_medium_woPos_deep6-regular-2026-04-13_t18-58-47'
    if model_id.endswith('.pth'):
        model_id = model_id[:-4]
    loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')
    print(model_id)
    print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}', end='\t')
    print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')
    # ---------------------------------------------------------------------------

    # Load full configuration (includes model, dataset, and training configs)
    config_path = f'{training_logs_path}/configs/{model_id}.json'
    with open(config_path, 'r') as f:
        config = json.load(f)
        
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)

    # Load trained weights
    model_path = f'{trained_models_path}/{model_id}.pth'
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    output = model(sample).detach().cpu().numpy()
    
    res_model = pd.DataFrame(
        {
            "x1": output[:, 0],
            "x2": output[:, 1],
            "x3": output[:, 2],
        }
    )
    res_model = pd.concat([locations, res_model], axis=1)

    figsize = (5, 5)
    fig = rgb(res_model, figsize=figsize)
    display(fig)
    plt.close(fig)
    time.sleep(2)

print('PCA:')
# example input (replace with your array)
X = sample.x.detach().cpu().numpy() # shape: (samples, features)
# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)
locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})
# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations, res_pca], axis=1)
fig = rgb(res_pca, figsize=figsize)    

In [ ]:
import pandas as pd
import json
import torch
import matplotlib.pyplot as plt
import plotly.express as px
import time
from IPython.display import display

# Loop
for i in [1,3,5,7,9,11,13,15]:
    model_id = files[i]
    if model_id.endswith('.pth'):
        model_id = model_id[:-4]

    loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')
    print(model_id)
    print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}', end='\t')
    print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')

    # ---------------------------------------------------------------------------
    # Load config
    config_path = f'{training_logs_path}/configs/{model_id}.json'
    with open(config_path, 'r') as f:
        config = json.load(f)

    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)

    # Load weights
    model_path = f'{trained_models_path}/{model_id}.pth'
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Forward pass
    output = model(sample).detach().cpu().numpy()

    # DataFrame
    res_model = pd.DataFrame({
        "x1": output[:, 0],
        "x2": output[:, 1],
        "x3": output[:, 2],
    })
    res_model = pd.concat([locations, res_model], axis=1)

    # ---------------------------------------------------------------------------
    '''
    # 1. Your existing plot
    figsize = (5, 5)
    fig = rgb(res_model, figsize=figsize)
    display(fig)
    plt.close(fig)
    '''
    
    # ---------------------------------------------------------------------------
    # 2. NEW: Interactive 3D scatter
    fig_3d = px.scatter_3d(
        res_model,
        x="x1",
        y="x2",
        z="x3",
        color="x3",  # optional coloring
        title=f"3D Scatter: {model_id}"
    )
    fig_3d.update_traces(marker=dict(size=1)) 
    fig_3d.show()

    time.sleep(2)

In [ ]:
for i in [0,2,4,6,8,10,12,14]:
    model_id = files[i]#'reg_medium_woPos_deep6-regular-2026-04-13_t18-58-47'
    if model_id.endswith('.pth'):
        model_id = model_id[:-4]
    loss_df = pd.read_csv(f'{training_logs_path}/loss_records/{model_id}.csv')
    print(model_id)
    print(f'Final training loss: {loss_df["loss"].iloc[-1]:.4f}', end='\t')
    print(f'Final validation loss: {loss_df["val_loss"].iloc[-1]:.4f}')
    # ---------------------------------------------------------------------------


    # Load full configuration (includes model, dataset, and training configs)
    config_path = f'{training_logs_path}/configs/{model_id}.json'
    with open(config_path, 'r') as f:
        config = json.load(f)
        
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)

    # Load trained weights
    model_path = f'{trained_models_path}/{model_id}.pth'
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    output = model(sample).detach().cpu().numpy()
    
    res_model = pd.DataFrame(
        {
            "x1": output[:, 0],
            "x2": output[:, 1],
            "x3": output[:, 2],
        }
    )
    res_model = pd.concat([locations, res_model], axis=1)

    figsize = (5, 5)
    fig = rgb(res_model, figsize=figsize)
    display(fig)
    plt.close(fig)
    time.sleep(2)

print('PCA:')
# example input (replace with your array)
X = sample.x.detach().cpu().numpy() # shape: (samples, features)
# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)
locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})
# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations, res_pca], axis=1)
fig = rgb(res_pca, figsize=figsize)    

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

# example input (replace with your array)
X = sample.x.detach().cpu().numpy() # shape: (samples, features)

# apply PCA to reduce to 3 dimensions
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})


# create pandas DataFrame
res_pca= pd.DataFrame(X_reduced, columns=["x1", "x2", "x3"])
res_pca = pd.concat([locations, res_pca], axis=1)

fig = rgb(res_pca, figsize=figsize)